In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk


In [2]:
#Universal Constants
c=3*10**8
mu_0 = 4*(np.pi)*(10**-7)
epsilon = 8.854*10**-12

### Predicting the Resonant Frequencies $f_0$ for 1-4mm 
The resonant frequency of a CPW resonator of $\frac{\lambda}{4}$ can be found using the followng equation[1]:
$$
f_0 = \frac{c}{\sqrt{\epsilon_{eff}}}\frac{1}{4l}
$$
where $c$ is the speed of light, $\epsilon_{eff}$ is th effective permitivity of CPW line and $l$ is length of the resonantor with a factor of $2$ as it is a half wavelength resonator which implies $2l = \lambda_0$, the fundamental mode. This uses the lumped element approximations to derive.

To calculate the the effective permeability the following equation is used[2].
$$
\epsilon_{eff} = \frac{\epsilon_r + 1}{2} + \frac{\epsilon_r - 1}{2} \frac{1}{\sqrt{1+12d/W}}
$$  
where $d$ is the height of the substrate and $W$ is the width of microstrip. For a microstrip line built on top a single substrate.


In [3]:
def find_effective_permeability(e_r,d,W):
    '''
    Determines the effective dielectric constant of a
    CPW when there is a single substrate
    Input:
    -   e_r = Dieletric Constant of the substrate 
    -   d   = Height of substrate
    -   W   = Width of CPW
    Output:
    -   e_e = Effective Dieletric Constant of CPW
    '''
    e_e = ((e_r+1)/(2))+((e_r-1)/(2))*(1/(np.sqrt(1+(12*d/W))))
    return e_e

def find_resonator_frequency_lumped_approx(e_r,d,W,l):
    '''
    Finds the resonant frequency of a CPW resonator defined
    on a single substrate using the lumped element approximation
    for a quarter wavelength resonator
    Input:
    -   e_r = Dielectric constant of the subtrate
    -   d   = Height of substrate
    -   W   = Width of CPW
    -   l   = Length of CPW resonator
    Output:
    -   f0  = Resonant Frequency of CPW resonantor
    '''
    e_e = find_effective_permeability(e_r,d,W)

    f0 = (c/(np.sqrt(e_e)))*(1/(4*l))
    return f0

theo_res_f = find_resonator_frequency_lumped_approx(11.6,270*(10**-6),10*(10**-6),12*(10**-3))
theo_res_f = theo_res_f/(10**9)
print(f'{theo_res_f} GHz')

2.433917419216113 GHz


In [4]:
#Resonator Lengths
l1 = 571.248*10**-6
l2 = 648.848*10**-6
l3 = 592.769*10**-6
l4 = 680.565*10**-6
l5 = 619.493*10**-6
l6 = 550.864*10**-6
l_vals = np.array([l1,l2,l3,l4,l5,l6])

#Substrate Height
h_s = 500*10**-6

#CPW width
w_CPW = 20*10**-6

#Sapphire dielectric constant
e_sap = 8.3

In [5]:
#Find Resoant Frequencies(GHz)
f0_lumped = (find_resonator_frequency_lumped_approx(e_sap,h_s,w_CPW,l_vals))/(10**9)
print(f0_lumped)

[59.55268494 52.43038765 57.39057232 49.98692581 54.91482901 61.75635396]


#### Conformal Mapping Techniques to find $f_0$
The phase velocity $v_p = c/\sqrt{\epsilon_eff}$ which is related to the resonant frequency $f_0$. 
$$
f_0 = \frac{v_p}{4l}
$$
$$
v_p = \frac{1}{\sqrt{L_lC_l}}
$$
Using conformal mapping techniques can be used to find the geometric contribution to $L_l$ and $C_l$.
$$
L_l = \frac{\mu_0}{4} \frac{K(k'_0)}{K(k_0)} \hspace{10mm} C_l = 4\epsilon_0\epsilon_{eff} \frac{K(k_0)}{K(k'_0)}
$$

Here, $K$ denotes the complete elliptic integral of the first kind with the arguments
$$
k_0 = \frac{w}{w+2s} \hspace{10mm} k'_0 = \sqrt{1-k^2_0}
$$


In [6]:
def find_f0_Conformal_Mapping(w,s,e_r,d,W,l):
    #Find Arguements of the First Kind
    k_0 = (w)/(w+2*s)
    k_prime = np.sqrt(1-k_0**2)

    #Find Effective Permeability
    e_eff = find_effective_permeability(e_r,d,W)

    #Elleptical Integrals
    L_l = ((mu_0)/(4))*(ellipk(k_prime))/(ellipk(k_0))
    C_l = 4*epsilon*e_eff*((ellipk(k_0))/(ellipk(k_prime)))

    #Find magnetic Inductance and total Capcitance per resonator
    L = L_l*l
    C = C_l*l

    #Find Resonant Frequencies
    f0 = ((1/np.sqrt(L_l*C_l))/(4*l))/(10**9)

    return L,C,f0


In [7]:
#Dimensions of CPW
width = 10*(10**-6)
seperation = 5*(10**-6)

#Apply Conformal Mapping
L_m,C,f0_CM = find_f0_Conformal_Mapping(width,seperation,e_sap,h_s,w_CPW,l_vals)

print('Resonant Frequencies using Conformal Mapping (GHz)')
print(f0_CM)
print('---------')
print('Resonant Frequencies using Lumped Approximation (GHz)')
print(f0_lumped)

Resonant Frequencies using Conformal Mapping (GHz)
[59.51211719 52.39467167 57.35147742 49.95287433 54.8774206  61.71428505]
---------
Resonant Frequencies using Lumped Approximation (GHz)
[59.55268494 52.43038765 57.39057232 49.98692581 54.91482901 61.75635396]


### Discussion
These values of the Resonant Frequencies are in agreement with each other as expected as both use the same value of the $\epsilon_{eff}$ to determine them and are in agreement in their values of $v_p$ the phase velocity. These values of the resonant frequency are valid assuming there is no contributions from kinetic inductance involved. Therefore an KeysightADS EM simulation of the circuit using PEC and the conductor and with no inputted is a valid method of verifying these theortical results.

### Introducing Kinetic Inductance
When a superconducting wire is suffciently thin, the kinetic inductance $L_K$ dominates over the magnetic inductance $L_m$ as calculated in conformal mapping. This occurs when the width $w$ and thickness $t$ of CPW resonantor $<<\lambda_L$ the london peneteration depth.
For the case of a homogenous superconducting cylinder of uniform circular or rectangular cross section. The kinetic inductance can be expressed as  
$$
L_K = \mu_0\lambda^2\frac{l}{A}
$$  
In this circuit design we will be using Niobium Nitride (NbN) at a thickness of 10nm. NbN $\lambda_L(0) = 230 \pm 20 \mathrm{nm}$ and $T_c = 14.8 \pm 0.3 K$. 